# Volleyball project — Pipeline
Load them with %run ./pipeline.ipynb ( in another jupyter notebook).

Basic working starting point, waiting Luka, Sean, Ben and Bisola  supply their final functions

## 1. Settings
Change paths here. Ball confidence is 0.35 and person confidence is 0.50. `frame_skip=1` tracks players on every frame; `2` tracks them every second frame. Ball detection still runs on every frame.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import shutil
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

CONFIG = {
    "ball_model": "models/ball_yolov8n.pt",
    "person_model": "models/person_yolov8n.pt",
    "ball_conf": 0.35, "player_conf": 0.50,
    "ball_class": None,  # Set the ball class ID if weights detect several classes.
    "tracker": "bytetrack.yaml", "imgsz": 640, "frame_skip": 1,
    "outputs_dir": "outputs",
    "events_function": None,  # Ben: function(players_df, ball_df, video_path, config)
    "stats_function": None,   # Bisola: function(players_df, ball_df, events_df)
}
PLAYER_COLUMNS = ["frame", "time_seconds", "player_id", "raw_track_id", "team","team_source", "confidence", "x1", "y1", "x2", "y2"]
BALL_COLUMNS = ["frame", "time_seconds", "x", "y", "confidence", "source"]
EVENT_COLUMNS = ["event_id", "frame", "time_seconds", "action", "team","player_id", "confidence", "source"]

## 2. Load models and check court corners
Use four points for Team A and four for Team B. Points must follow the edge of each polygon in clockwise or counterclockwise order, with no crossing lines. Coordinates must match the original video size. There is no click window inside the pipeline.

In [2]:
def load_models(config):
    for key in ["person_model", "ball_model"]:
        if not Path(config[key]).is_file():
            raise FileNotFoundError("Missing model: " + config[key])
    return {"players": YOLO(config["person_model"]),"ball": YOLO(config["ball_model"])}
def check_corners(court_corners, width, height):
    areas = {}
    for team in ["A", "B"]:
        points = np.asarray(court_corners[team], dtype=np.float32)
        if points.shape != (4, 2) or not np.isfinite(points).all():
            raise ValueError("Each team needs four valid (x, y) points.")
        if (points[:, 0] < 0).any() or (points[:, 0] >= width).any() or (points[:, 1] < 0).any() or (points[:, 1] >= height).any():
            raise ValueError("Court points must be inside the original frame.")
        if not cv2.isContourConvex(points) or cv2.contourArea(points) < 1:
            raise ValueError("Use four ordered corners forming a convex court area.")
        areas[team] = points
    return areas

## 3. Basic player trackin
The tracker gives IDs, not jersey numbers. The bottom-center of each box estimates the foot position. A point outside both courts gets `unknown`. The same tracker stays active during one run. A fresh run loads fresh models. Sean will improve IDs and teams later.

In [3]:
def assign_team(box, court_areas):
    x1, y1, x2, y2 = box
    foot = (float((x1 + x2) / 2), float(y2))
    for team, polygon in court_areas.items():
        if cv2.pointPolygonTest(polygon, foot, False) >= 0:
            return team
    return "unknown"
def detect_players(frame, model, config, court_areas, frame_number, fps):
    result = model.track(frame, persist=True, classes=[0], conf=config["player_conf"], imgsz=config["imgsz"], tracker=config["tracker"], verbose=False)[0]
    rows = []
    if result.boxes is None or result.boxes.id is None:
        return rows
    boxes = result.boxes.xyxy.cpu().numpy()
    ids = result.boxes.id.int().cpu().tolist()
    scores = result.boxes.conf.cpu().tolist()
    for box, track_id, score in zip(boxes, ids, scores):
        team = assign_team(box, court_areas)
        rows.append([frame_number, frame_number / fps, track_id, track_id, team, "court" if team != "unknown" else "unknown", float(score), *map(float, box)])
    return rows

## 4. Ball positions
Keep the highest confidence ball in each frame. If no ball is found, write a row with empty coordinates. A missing row has `source=detected` to show which stage produced it, but its coordinates and confidence are empty. Ben must ignore those rows. Luka will add smoothing and predicted positions later.

In [4]:
def detect_ball(frame, model, config, frame_number, fps):
    classes = None if config["ball_class"] is None else [config["ball_class"]]
    result = model.predict(frame, conf=config["ball_conf"], classes=classes, imgsz=config["imgsz"], verbose=False)[0]
    if result.boxes is None or len(result.boxes) == 0:
        return [frame_number, frame_number / fps, None, None, None, "detected"]
    best = int(result.boxes.conf.argmax().item())
    x1, y1, x2, y2 = result.boxes.xyxy[best].cpu().tolist()
    score = float(result.boxes.conf[best].item())
    return [frame_number, frame_number / fps, (x1+x2)/2, (y1+y2)/2, score, "detected"]

## 5. Connections for teammates
Without Ben’s function, events.csv has headers only. Without Bisola’s function, stats.json says pending. These are connection points, not finished action recognition or statistics. No scores or actions are guessed.

In [5]:
def detect_events(players_df, ball_df, video_path, config):
    function = config.get("events_function")
    if function is None:
        return pd.DataFrame(columns=EVENT_COLUMNS)
    events = function(players_df, ball_df, video_path, config)
    return events[EVENT_COLUMNS]
def compute_stats(players_df, ball_df, events_df, config):
    function = config.get("stats_function")
    if function is None:
        return {"status": "pending: connect statistics function", "team_totals": {}, "player_totals": {}, "rallies": []}
    return function(players_df, ball_df, events_df)
def draw_frame(frame, player_rows, ball_row, court_areas):
    image = frame.copy()
    for team, polygon in court_areas.items():
        cv2.polylines(image, [polygon.astype(np.int32)], True, (200, 200, 0), 2)
    for row in player_rows:
        x1, y1, x2, y2 = map(int, row[7:11])
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 200, 0), 2)
        cv2.putText(image, f"ID {row[2]} Team {row[4]}", (x1, max(20, y1-8)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1)
    if ball_row[2] is not None:
        cv2.circle(image, (int(ball_row[2]), int(ball_row[3])), 6, (0, 0, 255), -1)
    return image

## 6. Main function
Each run gets a new folder. Save one player row per detected player on processed frames, and one ball row per video frame. Skipped player frames have no player rows or boxes: old boxes are not copied forward. OpenCV keeps the original video FPS. The progress callback receives a number from 0 to 1.

The output video currently shows boxes, IDs, teams and ball positions. Ball trails and action labels need integration after the other stages arrive. The MP4 uses mp4v; Anthony will convert it to H.264 for browsers.

In [6]:
def run_pipeline(video_path, court_corners, config, progress=None):
    config = {**CONFIG, **config}
    skip = config["frame_skip"]
    if not isinstance(skip, int) or isinstance(skip, bool) or skip < 1:
        raise ValueError("frame_skip must be an integer of at least 1.")
    cap = cv2.VideoCapture(str(video_path))
    writer = None
    try:
        if not cap.isOpened():
            raise ValueError("Cannot open video: " + str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS)
        if not np.isfinite(fps) or fps <= 0:
            raise ValueError("Video has an invalid FPS.")
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        areas = check_corners(court_corners, width, height)
        models = load_models(config)
        run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        folder = Path(config["outputs_dir"]) / run_id
        folder.mkdir(parents=True, exist_ok=False)
        paths = {name: str((folder / name).resolve()) for name in ["players.csv", "ball.csv", "events.csv", "stats.json", "annotated.mp4", "run_config.json"]}
        saved_config = {k: v for k, v in config.items() if not callable(v)}
        saved_config.update({"video_path": str(video_path), "court_corners": {k: v.tolist() for k,v in areas.items()}, "events_connected": callable(config.get("events_function")), "stats_connected": callable(config.get("stats_function"))})
        Path(paths["run_config.json"]).write_text(json.dumps(saved_config, indent=2))
        writer = cv2.VideoWriter(paths["annotated.mp4"], cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
        if not writer.isOpened():
            raise RuntimeError("Cannot create output video.")
        players, balls = [], []
        number = 0
        if progress:
            progress(0.0)
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            rows = []
            if number % skip == 0:
                rows = detect_players(frame, models["players"], config, areas, number, fps)
                players.extend(rows)
            ball = detect_ball(frame, models["ball"], config, number, fps)
            balls.append(ball)
            writer.write(draw_frame(frame, rows, ball, areas))
            number += 1
            if progress and total > 0:
                progress(min(0.9 * number / total, 0.9))
        if number == 0:
            raise ValueError("Video contains no readable frames.")
        writer.release()
        writer = None
        players_df = pd.DataFrame(players, columns=PLAYER_COLUMNS)
        ball_df = pd.DataFrame(balls, columns=BALL_COLUMNS)
        players_df.to_csv(paths["players.csv"], index=False)
        ball_df.to_csv(paths["ball.csv"], index=False)
        events_df = detect_events(players_df, ball_df, str(video_path), config)
        events_df.to_csv(paths["events.csv"], index=False)
        stats = compute_stats(players_df, ball_df, events_df, config)
        Path(paths["stats.json"]).write_text(json.dumps(stats, indent=2))
        if progress:
            progress(1.0)
        return paths
    finally:
        cap.release()
        if writer is not None:
            writer.release()

## 7. UI stub
This separate function returns existing files, does not run detection.

In [7]:
def run_pipeline_stub(video_path, court_corners, config, progress=None):
    folder = Path(config["stub_folder"])
    names = ["players.csv", "ball.csv", "events.csv", "stats.json", "annotated.mp4", "run_config.json"]
    paths = {name: str((folder / name).resolve()) for name in names}
    for name, path in paths.items():
        if not Path(path).is_file():
            raise FileNotFoundError("Stub needs an existing file: " + path)
    if progress:
        progress(1.0)
    return paths

## 8. Run on video
Replace the example corners with the eight points from your video. Run this example manually after checking paths. It stays commented so importing this notebook does not start processing.

In [9]:
# court_corners = {"A": [(20, 100), (300, 100), (300, 400), (20, 400)], "B": [(320, 100), (600, 100), (600, 400), (320, 400)]}
# outputs = run_pipeline("input_videos/volleyball_sample.mp4", court_corners, CONFIG, progress=lambda p: print(round(p*100), "%"))
# print(outputs)

## 9. Add the final team functions
Copy only approved final functions into this notebook. Keep each person’s experiments in their own notebook. Adapt the function arguments when needed.

- Luka: replace basic ball detection and add ball smoothing.
- Sean: replace tracking and court-only team assignment. Keep raw IDs before stitching.
- Ben: connect `CONFIG["events_function"] = ben_detect_events`.
- Bisola: connect `CONFIG["stats_function"] = bisola_compute_stats`.
- Anthony: use the returned paths and progress callback in his UI.

After each merge, run the 41-second clip. Check that the video plays, ball.csv has all frames, IDs and teams are visible, and all six files exist. Processing every second frame changes tracker timing; compare IDs before using it for final results.

## Hai’s report section — simple draft
I prepared a setup notebook and a shared pipeline notebook. The settings are stored in one dictionary. The pipeline accepts a video and eight court points. It writes player positions, ball positions, events, statistics, an annotated video and run settings into a separate folder for each run. A progress callback lets the web page show progress.

The current version uses basic player tracking and court areas to assign teams. Action recognition and statistics need the final functions from the other team members. I have not yet measured speed or checked this version on the project video. I will add these results after testing. No new model training was done in my part.